# Step 3: exploratory data analysis

This notebook is deliberately thin. The logic lives in `src/eda.py`, so notebooks stay small and merge
cleanly (see CONTRIBUTING.md), and the same code produces the figures saved by `python -m src.eda`.

Run the cells from top to bottom. If a banner says SYNTHETIC, every number here comes from the
simulated dataset and describes nothing about Uttarakhand.

In [ ]:
import os, sys
from pathlib import Path

# Jupyter starts in the notebook folder; the project expects the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src import config, eda, viz

viz.apply_style()
df = eda.load()
df.head()

## 1. Class balance

How many landslide points against stable points. This decides two things: whether SMOTE is needed in
Step 4, and how much to trust accuracy. With a 1:2 split, a model that answers "stable" every time is
already about 67 percent accurate while being useless, which is why the report leads with AUC,
precision and recall.

In [ ]:
fig = eda.plot_class_balance(df)

## 2. Missing values

scikit-learn will not fit with gaps, so Step 4 fills them. The pattern matters more than the count:
values missing at the state edge usually mean a raster does not fully cover the boundary, while values
missing all over usually mean a CRS or extent mismatch in the QGIS extraction.

In [ ]:
eda.missing_report(df)

In [ ]:
fig = eda.plot_missing(df)

## 3. Correlation between factors

Two factors carrying the same information make the model unstable and add nothing. Anything near
plus or minus 0.8 here is a candidate for the VIF step in Step 4, which drops one of the pair.

The bottom row shows each factor against the label, which is a first look at what predicts landslides.
Read the colours as direction: blue is negative, red is positive, white is no linear relationship.

In [ ]:
fig = eda.plot_correlation(df)

## 4. Each factor, split by class

Where the two curves separate, the factor carries signal. Where they sit on top of each other, it
probably does not, at least on its own.

Watch for factors that separate in a non-linear way, for example risk peaking at middle slopes and
falling again on cliffs. A straight-line model cannot use that shape, which is the argument for the
RBF kernel in Step 5 and for Random Forest in Step 6.

In [ ]:
fig = eda.plot_distributions(df)

## 5. Categorical factors

For soil, rock type and land cover, the useful view is the share of points in each class that are
landslides, against the overall rate (the dashed line). Bars far from that line carry information.
Check the n on each bar: a class with very few points can look dramatic and mean nothing.

In [ ]:
fig = eda.plot_categorical_rates(df)

## 6. Which factors separate the classes best

Point-biserial correlation is Pearson correlation against the 0/1 label, so it ranks factors by how
strongly they track the outcome. Compare this ranking later with the Random Forest feature importance
from Step 6: they usually agree on the top few, and where they disagree the reason is interactions,
which a single correlation cannot see.

In [ ]:
eda.class_separation(df)

## 7. Data quality notes

Everything the checks flagged, in plain sentences. Anything marked IMPOSSIBLE or USELESS needs fixing
in QGIS before the models are trained.

In [ ]:
for note in eda.quality_checks(df):
    print(note)

## What this means for Step 4

1. Fill missing values: median for numbers, most frequent for categories.
2. Convert aspect before use. It is circular, so 359 and 1 degrees are neighbours that a model reads
   as opposites, and -1 marks flat ground with no direction at all.
3. Deal with any class flagged as a LEAK RISK above. A category that appears in only one of the two
   labels predicts perfectly here and not at all in the real world. The usual cause is our own
   sampling rule, such as water bodies being excluded from the stable points in Step 2e. Drop those
   rows or merge the class, and record which you chose.
4. Encode the three categorical columns as dummy variables, dropping one level per column so the
   dummies are not perfectly collinear.
5. Run VIF and drop anything above 10, one at a time, logging each drop.
6. Split 70/30 with stratification, then scale using the training set only.
7. Apply SMOTE to the training set only, never to the test set.

Steps 5 and 6 depend on that order. Scaling or resampling before the split leaks information from the
test set into training and inflates every score.